#### SBERT
- BERT 모델 : 문장 이해용 Encoder
    - 문장 쌍 비교
- SBERT 모델 : 문장 의미 임베딩
    - 벡터의 비교용

In [2]:
# !pip install sentence-transformers

In [3]:
import torch
from sentence_transformers import SentenceTransformer, util

c:\Users\johnh\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# 모델을 로드 -> 두개의 문장을 비교(코사인 유사도)
# 다목적 한국 SBERT
model_name = 'jhgan/ko-sroberta-multitask'
# 문장 유사도 특화
model_name2 = 'BM-K/KoSimCSE-roberta-multitask'

sbert = SentenceTransformer(model_name)
sbert2 = SentenceTransformer(model_name2)

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [5]:
# 최대 토론의 길이를 설정
sbert.max_seq_length =256
sbert2.max_seq_length = 256

In [6]:
doc1 = "이 카메라는 색감이 자연스럽고 배터리도 오래간다"
doc2 = "배터리 성능이 좋고 사진 품질이 뛰어나다"

In [7]:
# 두개의 문장을 임베딩 -> 코사인 유사도 계산
# sbert 인 경우
with torch.inference_mode():
    emb1 = sbert.encode(doc1, convert_to_tensor=True, normalize_embeddings=True)
    emb2 = sbert.encode(doc2, convert_to_tensor=True, normalize_embeddings=True)
# 코사인 유사도 계산
cos_sim = util.cos_sim(emb1, emb2).item()
print("유사도 : ", round(cos_sim, 4))

유사도 :  0.6948


In [8]:
# 두개의 문장을 임베딩 -> 코사인 유사도 계산
# sbert 인 경우
with torch.inference_mode():
    emb1 = sbert2.encode(doc1, convert_to_tensor=True, normalize_embeddings=True)
    emb2 = sbert2.encode(doc2, convert_to_tensor=True, normalize_embeddings=True)
# 코사인 유사도 계산
cos_sim2 = util.cos_sim(emb1, emb2).item()
print("유사도 : ", round(cos_sim2, 4))

유사도 :  0.734


In [9]:
emb1.shape

torch.Size([768])

In [10]:
sentences = [
    '삼성전자 주가가 올랐다',
    '코스피가 상승 마감했다',
    '비가 많이 와서 항공편이 취소됐다'
]

with torch.inference_mode():
    embs = sbert2.encode(sentences, convert_to_tensor=True, normalize_embeddings=True)

sim_metrix = util.cos_sim(embs, embs)

In [11]:
print(sim_metrix)

tensor([[ 1.0000,  0.3585, -0.0052],
        [ 0.3585,  1.0000,  0.1199],
        [-0.0052,  0.1199,  1.0000]])


In [12]:
new_sentence = '증시가 강세였다'
# 임베딩
new_emb = sbert2.encode(new_sentence, convert_to_numpy=True, normalize_embeddings=True)
# 유사도가 높은 상위의 n개 확인
top_n = 2
hits = torch.topk(
    util.cos_sim(new_emb, embs).squeeze(0), k = top_n
)
hits

torch.return_types.topk(
values=tensor([0.6161, 0.6014]),
indices=tensor([0, 1]))

In [13]:
for score, idx in zip(hits.values.tolist(), hits.indices.tolist() ):
    print(f"{sentences[idx]} | score {round(score, 3)}")

삼성전자 주가가 올랐다 | score 0.616
코스피가 상승 마감했다 | score 0.601


### 연습
- ratings_train.txt 파일 로드
- 결측치 제거
- documents 컬럼의 문자 정규화(특수문자 제거, 2칸 이상의 공백 제거, 좌우 공백 제거)
- 중복 documents 제거, 글자의 수가 1개 이하인 행은 제거
- DataFrame에서 sample(n = 10000, random_state = 42)로 임의의 데이터를 추출하여 저장(head() -> 상위 데이터 tail() -> 하위 데이터 | sample() -> 무작위 데이터)
- train, test 8:2 로 데이터 분할
- sbert 모델은 'BM-K/KoSimCSE-roberta-multitask' 을 이용
- Dataset을 정의 (Trainer 이용하지 않고 Dataset과 DataLoader 사용)
    - 입력받은 document 와 label을 document는 SBERT 모델을 이용하여 인코딩
    - label 데이터를 tensor형태로 변환
    - '__len__' 함수는 라벨의 길이를 되돌려준다
    - '__getitem__' 함수는 인코딩된 데이터[idx], label[idx]를 되돌려준다
- Dataset을 train,test를 이용하여 Dataset 생성
- DataLoader를 이용하여 배치의 사이즈는 128 shuffle 은 True로 구성한다.

In [14]:
import re
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, BertModel, Trainer, TrainingArguments

In [15]:
def normalize_token_text(text : str) -> str:
    text = re.sub(r'[^가-힣a-zA-Z0-9\s\.]', " ", text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [16]:
# 데이터 로드
df = pd.read_csv("../data/ratings_train.txt", sep='\t')
df.dropna(subset='document',inplace=True)
df['document'] = df['document'].map(normalize_token_text) # 정규화
df.drop_duplicates(subset=['document'] , inplace=True) # 중복데이터 제거
df = df.loc[df['document'].str.len() > 1] # document길이가 1이하면 제거
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 144637 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        144637 non-null  int64 
 1   document  144637 non-null  object
 2   label     144637 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 4.4+ MB


In [17]:
df.head()

,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화 스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [18]:
df2 = df.sample(n = 10000, random_state=42)
df2.reset_index(drop= True, inplace=True)
train_df, test_df = train_test_split(
    df2, test_size=0.2, random_state=42, stratify=df2['label']
)

In [19]:
model_name2

'BM-K/KoSimCSE-roberta-multitask'

In [20]:
MODEL_NAME = 'BM-K/KoSimCSE-roberta-multitask'
sbert3 = SentenceTransformer(MODEL_NAME)

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [21]:
from torch.utils.data import Dataset, DataLoader

In [22]:
# - Dataset을 정의 (Trainer 이용하지 않고 Dataset과 DataLoader 사용)
#     - '__init__(self, document, labels)'
#     - 입력받은 document 와 label을 document는 SBERT 모델을 이용하여 인코딩
#     - label 데이터를 tensor형태로 변환
#     - '__len__' 함수는 라벨의 길이를 되돌려준다
#     - '__getitem__' 함수는 인코딩된 데이터[idx], label[idx]를 되돌려준다
#     - 인코딩은 __init__ 함수 안에서 진행한다
class SBERTDataset(Dataset):
    def __init__(self, df, model_name):
        self.documents = df['document'].tolist()
        self.labels = torch.tensor(df['label'].tolist())
        self.sbert = SentenceTransformer(model_name)
        self.sbert.max_seq_length = 256
        with torch.inference_mode():
            self.embeddings = sbert3.encode(
                self.documents, convert_to_tensor=True, normalize_embeddings=True
            )
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.embeddings[idx], self.labels[idx]


In [23]:
# - Dataset을 train,test를 이용하여 Dataset 생성
train_dataset = SBERTDataset(train_df, MODEL_NAME)
test_dataset = SBERTDataset(test_df, MODEL_NAME)

No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.
No sentence-transformers model found with name BM-K/KoSimCSE-roberta-multitask. Creating a new one with mean pooling.


In [24]:
# - DataLoader를 이용하여 배치의 사이즈는 128 shuffle 은 True로 구성한다.
from torch.utils.data import DataLoader
train_dataloader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)
# test DataLoader
test_dataloader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)
# DataLoader 확인
for batch in train_dataloader:
    inputs, labels = batch
    print("입력 데이터 크기 : ", inputs.shape)
    print("라벨 데이터 크기 : ", labels.shape)
    break
# - Dataset을 train,test를 이용하여 Dataset 생성
# train 데이터셋의 크기
print("train 데이터셋의 크기 : ", len(train_dataset))
# test 데이터셋의 크기
print("test 데이터셋의 크기 : ", len(test_dataset))


입력 데이터 크기 :  torch.Size([128, 768])
라벨 데이터 크기 :  torch.Size([128])
train 데이터셋의 크기 :  8000
test 데이터셋의 크기 :  2000


In [28]:
class MLPHead(nn.Module):
    def __init__(self, input_dim, hidden = 256, num_classes = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden),
            nn.ReLU(), # 비선형의 구조를 이해
            nn.Dropout(0.2),
            nn.Linear(hidden, num_classes)
        )
    
    def forward(self, x):
        result = self.net(x)
        return result

In [29]:
# MLPHead class를 생성하기 위해 input_dim 매개변수에 인자는 필수 항목
# input_dim -> sbert3 모델에서 임베딩이 된 독립변수의 피쳐의 수
# 입력데이터 -> sbert3모델에서 임베딩이 된 독립 변수의 피쳐의 수
# sbert3 에서 설정이 된 출력 피쳐의 수를 변수에 저장
in_dim = sbert3.get_sentence_embedding_dimension() # 출력 피쳐의 수를 되돌려주는 내장함수
in_dim

768

In [30]:
clf = MLPHead(in_dim)
# 손실함수 -> 예측값과 실제값의 차이를 확인하는 함수
crit = nn.CrossEntropyLoss()
# 옵티마이저
opt = torch.optim.Adam(clf.parameters(), lr=2e-4)

In [32]:
clf.train()

for epoch in range(5):
    total = 0.0
    for x,y in train_dataloader:
        # x : document 데이터가 임베딩 벡터가 된 묶음
        # y : labels 데이터가 tensor형태 묶음
        opt.zero_grad()
        logits = clf(x)
        # 손실 계산()
        loss = crit(logits, y)
        # 역전파
        loss.backward()
        # 스탭
        opt.step()
        total += loss.item() * x.size(0)
    print(f"epoch : {epoch}, loss : {total/len(train_dataset)}")



epoch : 0, loss : 0.6310898303985596
epoch : 1, loss : 0.49396160078048706
epoch : 2, loss : 0.439065461397171
epoch : 3, loss : 0.4216532127857208
epoch : 4, loss : 0.4113210973739624


In [33]:
# 테스트 데이터를 이용하여 정확도, f1_score 확인
clf.eval()

y_true, y_pred = [], []

with torch.inference_mode():
    for x, y, in test_dataloader:
        logits = clf(x) # 예측 데이터 -> [0.xxx, 0.xxxx]
        pred = logits.argmax(dim=1).tolist()
        # y_true에 y를 list형태로 변환하고 데이터를 확장시킨다
        y_true.extend(y.tolist())
        y_pred += pred
print(y_true)
print(y_pred)

[0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 

In [34]:
print('accuracy_score', accuracy_score(y_true, y_pred))
print('f1_score', f1_score(y_true, y_pred))

accuracy_score 0.813
f1_score 0.8122489959839357


In [42]:
samples = [
    '와... 영화 진짜 최고였습니다. 또 보고싶습니다.',
    '스토리가 엉망이고 연기도 별로였다. 추천할만한 영화는 아니다.',
    '그럭저럭 본만했지만 크게 인상적이지는 않았다.'
]

id2label = {
    0 : '부정',
    1 : '긍정'
}
# 예측값을 되돌려주는 함수
@torch.no_grad()
def predict_review(
    texts,
    batch_size = 128
):
    if isinstance(texts, str):
        texts = [texts]

    texts_norm = [normalize_token_text(t) for t in texts]

    sbert3.eval()
    clf.eval()

    result = []

    # 배치 데이터로 구성 -> encode -> 분류 모델에 데이터 입력 -> 출력 값을 설정
    for idx in range(0, len(texts_norm), batch_size):
        batch_texts = texts_norm[ idx: idx+batch_size]
        embs = sbert3.encode(
            batch_texts,
            convert_to_tensor=True,
            normalize_embeddings=True
        )

        logits = clf(embs)
        probs = logits.softmax(dim = -1)
        preds = probs.argmax(dim = -1).tolist()

        for idx2, pred in enumerate(preds):
            # idx2 : 인덱스
            # pred : 예측 값(예측 확률의 인덱스 : 확률이 높은 곳의 인덱스(0,1))
            # 높은 예측율
            prob = float(probs[idx2, pred])

            review = texts[idx+idx2]

            labels = id2label[pred]

            result.append(
                {
                    'text' : review,
                    'prob' : prob,
                    'label' : labels
                }
            )

        return result

In [43]:
out_data = predict_review(samples)

In [44]:
out_data

[{'text': '와... 영화 진짜 최고였습니다. 또 보고싶습니다.',
  'prob': 0.9768144488334656,
  'label': '긍정'},
 {'text': '스토리가 엉망이고 연기도 별로였다. 추천할만한 영화는 아니다.',
  'prob': 0.9641687870025635,
  'label': '부정'},
 {'text': '그럭저럭 본만했지만 크게 인상적이지는 않았다.',
  'prob': 0.7967485189437866,
  'label': '부정'}]

### 복습
- 가전 폴더 안에 모든 데이터파일을 로드해서 하나의 데이터프레임으로 생성
- 감정에 대한 데이터들이 3개 분류 -> 2개의 분류로 변경 (부정, 중립 -부정)
- 감정 데이터가 없는 데이터들은 따로 저장
- train, test 비율은 8:2
- Dataset을 구성할 때 생성자 함수에서는 데이터를 그냥 self 변수에 저장
- '__getitem__' 함수에서 임베딩 후 되돌려주는 형태를 구성 변경
- RawText 데이터를 이용하여 감정분석 모델을 생성
- SBERT 모델을 이용하여 임베딩
- 다중퍼셉트론 모델을 이용하여 감정 분석 (Linear -> ReLU -> DropOut -> Linear)
- 검증 데이터를 이용하여 정확도와 f1_score 확인
- 감정 데이터가 없는 RawText에서 sample(10)를 출력하여 감정 예측

- 다중 퍼셉트론 모델이 아닌 머신러닝 모델 (SVC)을 이용하여 감정 분석 예측